<a href="https://colab.research.google.com/github/alireza1420/stargazer-prediction-pipeline/blob/carolines-neural-net/Best_Model_Caroline_23_May.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.feature_selection import SelectKBest, f_regression

csv_url = "https://raw.githubusercontent.com/alireza1420/stargazer-prediction-pipeline/main/data_preparation_new/github_repo_features_new.csv"
df = pd.read_csv(csv_url)

df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)
ref_date = datetime(2025, 5, 1)

df["project_age"] = (ref_date - df["created_at"]).dt.days
df["days_since_update"] = (ref_date - df["updated_at"]).dt.days
df["days_since_push"] = (ref_date - df["pushed_at"]).dt.days

df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
df["update_rate"] = 1 / (1 + df["days_since_update"])

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

for col in ["has_wiki", "has_projects", "has_downloads", "is_fork", "archived"]:
    df[col] = df[col].astype(int)

features = [
    'open_issues', 'size', 'has_wiki', 'has_projects', 'has_downloads',
    'is_fork', 'archived', 'language', 'license', 'subscribers_count',
    'contributors_count', 'commits_count', 'readme_size', 'project_age',
    'days_since_update', 'days_since_push', 'forks_per_day', 'update_rate'
]

X = df[features]
y = df["stars"]

#preprocessing
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_regression, k=15)),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

#spliting  and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

#Evaluate
predictions = pipeline.predict(X_test)
print("R2 score on test set:", r2_score(y_test, predictions))


R2 score on test set: 0.6825094880204481


In [8]:
# Create a real-world sample repository
sample_repo = {
    'name': 'ml-web-app',
    'full_name': 'data-scientist/ml-web-app',
    'created_at': '2023-05-10T08:00:00Z',
    'updated_at': '2023-11-15T14:25:00Z',
    'pushed_at': '2023-11-15T14:30:00Z',
    'language': 'Python',  # Must be in encoders['language'].classes_
    'license': 'mit',      # Must be in encoders['license'].classes_
    'forks': 87,
    'watchers': 420, # same as stars, unknown
    'open_issues': 12,
    'size': 3500,
    'has_wiki': True,
    'has_projects': False,
    'has_downloads': True,
    'is_fork': False,
    'archived': False,
    'subscribers_count': 150,
    'readme_size': 1024,
    'commits_count': 85,
    'contributors_count': 12
}


# Load the Keras model
loaded_model = tf.keras.models.load_model("neural_network_new_model.h5", custom_objects={'mse': tf.keras.losses.MeanSquaredError()})

sample_df = pd.DataFrame([sample_repo])

sample_df.rename(columns={'watchers': 'stars'}, inplace=True)

sample_processed, _ = preprocessor.transform(sample_df)

sample_transformed = preprocessor_pipeline.transform(sample_processed)

predicted_stars = loaded_model.predict(sample_transformed)

print("Predicted number of stars for sample repo:", predicted_stars[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
Predicted number of stars for sample repo: 288263.8
